In [ ]:
import os
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    average_precision_score,
    f1_score,
    confusion_matrix,
    classification_report,
    precision_recall_fscore_support
)

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)

# from google.colab import drive
# drive.mount('/content/drive')

DATASET = "/kaggle/input/datasets/ang3loliveira/malware-analysis-datasets-api-call-sequences/dynamic_api_call_sequence_per_malware_100_0_306.csv"
CHECKPOINT_DIR = "/kaggle/working/checkpoints"
MODEL_DIR = "/kaggle/working/"

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

NUM_API_CALLS = 307
SEQ_LEN = 100
LATENT_DIM = 128
EMB_DIM = 128
BATCH_SIZE = 128
EPOCHS = 50
SEEDS = [10, 20, 30, 40, 50]


def seq_to_graph(seq_batch, is_gumbel=False):
    B = seq_batch.size(0)
    
    if not is_gumbel:
        src = seq_batch[:, :-1]
        dst = seq_batch[:, 1:]
        
        adj = torch.zeros((B, NUM_API_CALLS, NUM_API_CALLS), device=seq_batch.device)
        batch_index = torch.arange(B, device=seq_batch.device).unsqueeze(1)
        adj[batch_index, src, dst] += 1.0
        
        X = F.one_hot(seq_batch, NUM_API_CALLS).float().permute(0, 2, 1)
    else:
        X_curr = seq_batch[:, :-1, :] # [B, SEQ_LEN-1, NUM_API_CALLS]
        X_next = seq_batch[:, 1:, :]  # [B, SEQ_LEN-1, NUM_API_CALLS]
        adj = torch.bmm(X_curr.transpose(1, 2), X_next) 
        X = seq_batch.permute(0, 2, 1) 
    return adj, X


class MalwareSequenceDataset(Dataset):
    def __init__(self, dataframe):
        self.raw_seqs = dataframe.drop(columns=["hash", "malware"]).values
        self.y = dataframe["malware"].values

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        seq = torch.tensor(self.raw_seqs[idx], dtype=torch.long)
        label = torch.tensor(self.y[idx], dtype=torch.long)
        
        src = seq[:-1]
        dst = seq[1:]
        
        adj = torch.zeros((NUM_API_CALLS, NUM_API_CALLS), dtype=torch.float32)
        adj[src, dst] += 1.0
        
        X = F.one_hot(seq, NUM_API_CALLS).float().permute(1, 0)
        
        return adj, X, label


df = pd.read_csv(DATASET)

train_df, test_df = train_test_split(
    df,
    test_size=0.30,
    stratify=df["malware"],
    random_state=42
)

print("Train rows:", len(train_df))
print("Test rows :", len(test_df))

train_dataset = MalwareSequenceDataset(train_df)
test_dataset = MalwareSequenceDataset(test_df)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True if torch.cuda.is_available() else False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

class GATLayer(nn.Module):
    def __init__(self, in_features, out_features, heads=4):
        super().__init__()
        self.heads = heads
        self.out_features = out_features

        self.W = nn.Parameter(torch.randn(heads, in_features, out_features) * 0.01)
        self.a_src = nn.Parameter(torch.randn(heads, out_features, 1) * 0.01)
        self.a_dst = nn.Parameter(torch.randn(heads, out_features, 1) * 0.01)

    def forward(self, adj, X):
        B, N, _ = X.size() 
        
        Wh = torch.matmul(X.unsqueeze(1), self.W) 
        f1 = torch.matmul(Wh, self.a_src) 
        f2 = torch.matmul(Wh, self.a_dst) 
        
        e = F.leaky_relu(f1 + f2.transpose(-2, -1))
        mask = torch.where(adj.unsqueeze(1) > 0, e, torch.full_like(e, -9e15))
        attention = F.softmax(mask, dim=-1)

        h_out = torch.matmul(attention, Wh)
        h_out = h_out.permute(0, 2, 1, 3).reshape(B, N, self.heads * self.out_features)
        return h_out

class GATClassifier(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.gat1 = GATLayer(in_features=SEQ_LEN, out_features=64, heads=4)
        self.gat2 = GATLayer(in_features=64 * 4, out_features=32, heads=4)
        self.dropout = nn.Dropout(0.5)
        self.fc = nn.Linear(NUM_API_CALLS * 32 * 4, num_classes)

    def forward(self, adj, X, return_features=False):
        Z = F.elu(self.gat1(adj, X))
        Z = F.elu(self.gat2(adj, Z))
        Z = self.dropout(Z)
        
        features = Z.reshape(Z.size(0), -1)
        logits = self.fc(features)
        
        if return_features:
            return logits, features
        return logits

class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        self.init_fc = nn.Linear(LATENT_DIM, 256)
        self.rnn = nn.GRU(input_size=EMB_DIM, hidden_size=256, batch_first=True)
        self.token_proj = nn.Linear(256, NUM_API_CALLS)
        self.start_token = nn.Parameter(torch.zeros(1, 1, EMB_DIM))

    def forward(self, z):
        B = z.size(0)
        h0 = torch.tanh(self.init_fc(z)).unsqueeze(0)
        inp = self.start_token.repeat(B, SEQ_LEN, 1)
        out, _ = self.rnn(inp, h0)
        logits = self.token_proj(out)
        
        probs = F.gumbel_softmax(logits, tau=0.5, hard=False)
        return probs 

class HybridSGAN(nn.Module):
    def __init__(self, G, D):
        super().__init__()
        self.G = G
        self.D = D

    def forward(self, adj, X):
        return self.D(adj, X)

def evaluate(model, test_loader):
    model.eval()
    probs_all, labels_all = [], []

    with torch.no_grad():
        for adj, X, labels in test_loader:
            adj = adj.to(DEVICE)
            X = X.to(DEVICE)
            
            logits = model(adj, X)
            probs = torch.softmax(logits[:, :2], dim=1)
            
            probs_all.extend(probs[:, 1].cpu().numpy())
            labels_all.extend(labels.numpy())

    labels_all = np.array(labels_all)
    probs_all = np.array(probs_all)
    preds = (probs_all > 0.5).astype(int)
    
    cm = confusion_matrix(labels_all, preds)
    tn, fp, fn, tp = cm.ravel()

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels_all, preds, labels=[0, 1], zero_division=0
    )

    return {
        "accuracy": accuracy_score(labels_all, preds),
        "roc_auc": roc_auc_score(labels_all, probs_all),
        "pr_auc": average_precision_score(labels_all, probs_all),
        "macro_f1": f1_score(labels_all, preds, average="macro"),
        "weighted_f1": f1_score(labels_all, preds, average="weighted"),
        "fpr": fp / (fp + tn),
        "c0_precision": precision[0],
        "c0_recall": recall[0],
        "c0_f1": f1[0],
        "c1_precision": precision[1],
        "c1_recall": recall[1],
        "c1_f1": f1[1],
        "report": classification_report(labels_all, preds, digits=4, zero_division=0)
    }

gat_results = []

for seed in SEEDS:
    print("\nSeed:", seed)
    set_seed(seed)

    classifier = GATClassifier(num_classes=2).to(DEVICE)
    optimizer = optim.Adam(classifier.parameters(), lr=2e-4)

    for epoch in range(EPOCHS):
        classifier.train()
        for adj, X, labels in train_loader:
            adj = adj.to(DEVICE)
            X = X.to(DEVICE)
            labels = labels.to(DEVICE)

            logits = classifier(adj, X)
            loss = F.cross_entropy(logits, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        print(f"epoch {epoch+1}/{EPOCHS}...")

    metrics = evaluate(classifier, test_loader)
    
    torch.save(classifier, os.path.join(MODEL_DIR, f"gat_seed_{seed}.pt"))
    gat_results.append({"seed": seed, **metrics})


def summarize(results, name):
    df_res = pd.DataFrame(results)
    print("\n" + "="*60 + f"\n{name}\n" + "="*60)

    metrics = [
        "accuracy", "roc_auc", "pr_auc", "macro_f1", "weighted_f1", "fpr",
        "c0_precision", "c0_recall", "c0_f1",
        "c1_precision", "c1_recall", "c1_f1"
    ]
    
    print("Global & Structural Mean Metrics:")
    for m in metrics:
        mean = df_res[m].mean()
        std = df_res[m].std()
        print(f"  {m}: {mean:.4f} ± {std:.4f}")

    print("\nPer Seed Granular Breakdown:")
    print(df_res[["seed"] + metrics].to_string(index=False))
    return df_res

gat_df = summarize(gat_results, "GAT BASELINE")
gat_df.to_csv("/content/gat_results.csv", index=False)

In [ ]:
import os
import torch
import pandas as pd

SEEDS = [10, 20, 30, 40, 50]

gat_results = []

for seed in SEEDS:

    model_path = os.path.join(
        "/kaggle/working/",
        f"gat_seed_{seed}.pth"
    )

    if not os.path.exists(model_path):
        print(f"Missing: {model_path}")
        continue

    print(f"\nLoading seed {seed}")

    D = GATClassifier(num_classes=3).to(DEVICE)
    D.load_state_dict(torch.load(model_path, map_location=DEVICE))
    D.eval()
    
    metrics = evaluate(D, test_loader)
    
    gat_results.append({
        "seed": seed,
        **metrics
    })

gat_df = pd.DataFrame(gat_results)

metrics = [
    "accuracy", "roc_auc", "pr_auc",
    "macro_f1", "weighted_f1", "fpr",
    "c0_precision", "c0_recall", "c0_f1",
    "c1_precision", "c1_recall", "c1_f1"
]

print("\n" + "="*80)
print("GAT")
print("="*80)

print("\nPer Seed Results:")
print(gat_df[["seed"] + metrics].to_string(index=False))

print("\nMean ± Std:")
for m in metrics:
    print(
        f"{m:15s}: "
        f"{gat_df[m].mean():.4f} ± {gat_df[m].std():.4f}"
    )